# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates loading, exploration, and processing of the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library, strictly referencing data entities using their `@id` values as defined by the Croissant schema.

### Dataset Source
Schema URL: [https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)

This dataset contains tabular records for 77 cancer survivors with second primary colorectal cancer, including clinical, pathologic, and molecular variables suitable for reproducible ML and clinical insight pipelines.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load the metadata and record set definitions using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the Croissant dataset package
dataset = mlc.Dataset(croissant_url)

# Print dataset-level metadata
meta = dataset.metadata
print(f"{meta.name}: {meta.description}")
print(f"Identifier: {meta.identifier}")
print(f"Published: {meta.datePublished}")
print(f"Version: {meta.version}")
print(f"License: {meta.license}")
print()

## 2. Data Overview
List all available record sets, each field and column, and their `@id` identifiers. This allows referencing any dataset component unambiguously.

In [ ]:
# List all record sets defined in the dataset
record_set_overview = []
for record_set in dataset.record_sets:
    info = {
        '@id': record_set.id,
        'name': getattr(record_set, 'name', None),
        'num_fields': len(record_set.fields),
        'description': getattr(record_set, 'description', ''),
        'fields': [f'@id: {fld.id}, name: {getattr(fld, "name", "")} (type: {getattr(fld, "data_type", "")})' for fld in record_set.fields]
    }
    record_set_overview.append(info)

for rs in record_set_overview:
    print(f"Record Set @id: {rs['@id']}")
    print(f"  Name: {rs['name']}")
    print(f"  Description: {rs['description']}")
    print(f"  Fields ({rs['num_fields']}):")
    for i, field in enumerate(rs['fields']):
        print(f"    [{i+1}] {field}")
    print("\n---\n")

# Keep a list of record set @ids for later
record_set_ids = [rs['@id'] for rs in record_set_overview]

## 3. Data Extraction
Fetch data for each record set into a Pandas DataFrame. Use record set and field `@id`s from the Data Overview above.

In [ ]:
# Load all record sets' records into dataframes (by `@id`)
dataframes = {}
for record_set_id in record_set_ids:
    records_iter = dataset.records(record_set=record_set_id)
    records = list(records_iter)
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df

# Print an overview of each dataframe's columns (by field @id)
for record_set_id, df in dataframes.items():
    print(f"Record set @id: {record_set_id}")
    print(f"Columns (fields' @id): {df.columns.tolist()}")
    print(df.head(), "\n---\n")

## 4. Exploratory Data Analysis (EDA)
Apply common processing steps using the columns' `@id` values. Typical operations: filtering, normalization, grouping.

In [ ]:
# For demonstration, select the first tabular record set
if dataframes:
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]
    
    # Identify a likely numeric field (@id by convention, usually age, interval, or count)
    numeric_col = None
    for col in df.columns:
        # Heuristic: columns with all int/float or named age/interval
        if pd.api.types.is_numeric_dtype(df[col]) or 'age' in col.lower() or 'interval' in col.lower():
            numeric_col = col
            break
    
    if numeric_col:
        print(f"Selected numeric field: {numeric_col}")
        print(f"Summary statistics:\n{df[numeric_col].describe()}")
        
        # Set threshold for filtering (e.g. one std above the mean if appropriate)
        threshold = df[numeric_col].mean()
        filtered_df = df[df[numeric_col] > threshold].copy()
        print(f"Filtered records where {numeric_col} > {threshold:.2f}:")
        print(filtered_df[[numeric_col]].head())
        
        # Normalize chosen field
        col_norm = numeric_col + '_normalized'
        filtered_df[col_norm] = (filtered_df[numeric_col] - filtered_df[numeric_col].mean()) / filtered_df[numeric_col].std()
        print(f"Normalized {numeric_col} (first 5 rows):")
        print(filtered_df[[numeric_col, col_norm]].head())
        
        # Select a categorical field (@id) for grouping (heuristically pick a non-numeric with <20 unique values)
        group_field = None
        for candidate in df.columns:
            if candidate != numeric_col and df[candidate].dtype == object and df[candidate].nunique() < 20:
                group_field = candidate
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_col].mean()
            print(f"Grouped mean of {numeric_col} by {group_field} (first 5 groups):")
            print(grouped_df.head())
    else:
        print("No numeric field detected for EDA in this record set.")
else:
    print("No tabular record sets loaded.")

## 5. Visualization
Visualize the distribution of a numeric variable and its grouping by a selected category, all referenced by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Reuse variables from above if available
if dataframes and 'numeric_col' in locals() and numeric_col:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_col], kde=True)
    plt.title(f'Distribution of {numeric_col} (@id)')
    plt.xlabel(numeric_col)
    plt.ylabel('Count')
    plt.tight_layout()
    plt.show()
    
    if 'group_field' in locals() and group_field:
        plt.figure(figsize=(10, 4))
        sns.boxplot(x=group_field, y=numeric_col, data=df)
        plt.title(f'{numeric_col} by {group_field} (@id)')
        plt.xlabel(group_field)
        plt.ylabel(numeric_col)
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
else:
    print("Nothing to visualize: no numeric field or dataframe available.")

## 6. Conclusion

- This notebook has demonstrated how to load, inspect, and analyze a clinical dataset via its Croissant schema using the `mlcroissant` library.  
- Data entities are referenced exclusively by their `@id`, enabling robust, schema-compliant workflows.  
- Standard EDA steps were performed, and visualizations generated for a selected numeric field and grouping, facilitating downstream clinical or ML modeling tasks.

_For further analysis or modeling, continue to use the dataframes and entity `@id` references established above._